In [ ]:
import numpy as np
import xarray as xr
import scipy.sparse as sp
import scipy.sparse.linalg as spla

In [ ]:

def solve_w_elliptic_with_product_rule(f_da, N2_arr, u_x_arr, b_x_arr, x_coords, z_coords,
                                       tol=1e-12, use_iterative=True, N2_floor=1e-10):
    """
    Solve N2 * w_xx + f^2 * w_zz = RHS
    with RHS = -2 * d/dx(u_x * b_x)  (full product derivative)
    BCs: w=0 at top/bottom, dw/dx=0 at left/right.
    Handles NaNs and small N2 floor.
    Returns w (nz, nx) in same ordering as inputs.
    """
    # ---- arrays ----
    f_vec = np.asarray(f_da)
    N2 = np.asarray(N2_arr).astype(float)
    u_x = np.asarray(u_x_arr).astype(float)
    b_x = np.asarray(b_x_arr).astype(float)
    x = np.asarray(x_coords)
    z = np.asarray(z_coords)

    nz, nx = N2.shape
    assert u_x.shape == (nz, nx)
    assert b_x.shape == (nz, nx)
    assert f_vec.size == nx

    # --- protective floors & NaN handling ---
    N2 = np.nan_to_num(N2, nan=N2_floor)
    N2[N2 < N2_floor] = N2_floor

    # --- compute RHS with product rule along x ---
    dx = np.gradient(x)  # spacing along x
    u_xx = np.gradient(u_x, x, axis=1)  # u_xx
    b_xx = np.gradient(b_x, x, axis=1)  # b_xx

    # (u_x * b_x)_x = u_xx * b_x + u_x * b_xx
    rhs_prod = u_xx * b_x + u_x * b_xx
    rhs = -2.0 * rhs_prod  # include the -2 factor

    # flatten RHS to bvec later
    bvec = rhs.flatten()

    # --- build sparse matrix A for N2*w_xx + f^2*w_zz ---
    Npts = nz * nx
    rows, cols, data = [], [], []

    def idx(i, j): return j * nx + i   # i: x, j: z

    # validity mask
    valid = np.isfinite(N2) & np.isfinite(rhs)

    for j in range(nz):
        for i in range(nx):
            p = idx(i, j)

            # masked or Dirichlet top/bottom -> w = 0
            if (not valid[j, i]) or j == 0 or j == nz-1:
                rows.append(p); cols.append(p); data.append(1.0)
                bvec[p] = 0.0
                continue

            N2c = N2[j, i]
            f2c = f_vec[i]**2

            # horizontal spacings
            dxm = x[i] - x[i-1] if i > 0 else x[1] - x[0]
            dxp = x[i+1] - x[i] if i < nx-1 else x[-1] - x[-2]

            # vertical spacings
            dzm = z[j] - z[j-1]
            dzp = z[j+1] - z[j]

            # x-coeffs (non-uniform spacing)
            if 0 < i < nx-1:
                axm = N2c / (dxm * (dxm + dxp))
                axp = N2c / (dxp * (dxm + dxp))
                axc = -(axm + axp)
            elif i == 0:
                axm = 0.0
                axp = 2.0 * N2c / (dxp**2)
                axc = -axp
            else:  # i == nx-1
                axp = 0.0
                axm = 2.0 * N2c / (dxm**2)
                axc = -axm

            # z-coeffs (interior guaranteed because top/bottom handled)
            azm = f2c / (dzm * (dzm + dzp))
            azp = f2c / (dzp * (dzm + dzp))
            azc = -(azm + azp)

            center = axc + azc

            # fill entries
            rows.append(p); cols.append(p); data.append(center)
            if i > 0:
                rows.append(p); cols.append(idx(i-1, j)); data.append(axm)
            if i < nx-1:
                rows.append(p); cols.append(idx(i+1, j)); data.append(axp)
            rows.append(p); cols.append(idx(i, j-1)); data.append(azm)
            rows.append(p); cols.append(idx(i, j+1)); data.append(azp)

    A = sp.csr_matrix((data, (rows, cols)), shape=(Npts, Npts))

    # --- solve ---
    try:
        if use_iterative:
            M = spla.spilu(A.tocsc(), drop_tol=1e-4)
            precond = spla.LinearOperator((Npts, Npts), lambda x: M.solve(x))
            w_flat, info = spla.bicgstab(A, bvec, tol=tol, maxiter=5000, M=precond)
            if info != 0:
                w_flat = spla.spsolve(A.tocsc(), bvec)
        else:
            w_flat = spla.spsolve(A.tocsc(), bvec)
    except Exception:
        w_flat = spla.spsolve(A.tocsc(), bvec)

    w = w_flat.reshape((nz, nx))
    # enforce masked and Dirichlet
    w[~valid] = 0.0
    w[0, :] = 0.0
    w[-1, :] = 0.0

    return w


In [ ]:
# convert coords & arrays to numpy
x_m = ds_glider.distance.values*1000   # distance in meters
z_m = ds_glider.ctd_depth.values # depth values in meters
f_vec = f_y.values                # coriolis parameter

N2 = (np.diff(b, axis=0)/-0.5) # b is buoyancy

zero_row = np.zeros((1, N2.shape[1]))  # Create a row of zeros with shape (1, 610)

# Concatenate the zero row to the top of N2
N2 = np.vstack([zero_row, N2])
u = gv_ref_bottom_smoothed  # geosrtophic velocity
ux = np.gradient(u, 1500.0, axis=1)

# make sure u_x, b_x, N2 have shape (nz, nx)
u_x_arr =  ux 
b_x_arr = by_smoothed # horizontal buoyancy gradient
N2_arr = N2  


# shape (nx,)
w_sol = solve_w_elliptic_with_product_rule(f_vec, N2_arr, u_x_arr, b_x_arr, x_m, z_m)
w_da_u = xr.DataArray(w_sol, coords={'ctd_depth': z_m, 'distance': x_m}, dims=['ctd_depth','distance'])